[Lab README](README.md)

# Lab 6.1: Inspectable Neo4j memory (optional)

This notebook keeps one memory story deliberately small: persist a preference statement, recall it for the same actor in a later session, prove a second actor with a conflicting preference about the same hotel gets their own row and never the first actor's, and inspect where each preference came from and which real `Hotel` it describes.

Two boundaries are explicit. Multi-tenant mode rejects memory writes that omit a user identifier, but the application must still authenticate actors and authorize session IDs. The library's semantic searches are store-wide in 0.5.0, so this lab does not use them as an isolation boundary. Its recall query starts at the selected `User`.

## Why memory in the graph rather than a managed store

A managed memory service is the shorter path to a working agent, and for many applications it is the right one. What it does not give you is the last section of this notebook. When a preference is a node in the same graph as your hotels, you can ask which message produced it, which `Hotel` it refers to, and which other memories point at that same hotel, and get an answer as a traversal rather than as a similarity score. That is the difference between memory you can query and memory you can only retrieve from. It is the same argument the rest of this workshop makes about hotel knowledge, applied to what the agent remembers, and it costs you the operational convenience of a service you do not run.

**Prerequisites:** Lab 1 has created exactly one `Hotel` named `AnyCompany Cairo Nile View`; Neo4j credentials point at the correct database; and AWS credentials in `AWS_REGION` may invoke Titan Text Embeddings V2. This lab uses the same Neo4j instance and credentials as Lab 1, so the repo-root `.env` described in the top-level README covers it. `load_config` reads this folder's `.env` first, then the repo-root `.env`. Section 1 checks the Titan model before anything is written. Without credentials, every live cell skips cleanly.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the two lines below first.
# !pip install -e ../workshop
# !pip install "neo4j-agent-memory[bedrock]==0.5.0"

print("Environment ready")

## 1. Configure one isolated workshop run

Every actor and session identifier includes a short run ID, so rerunning the notebook cannot append to an earlier transcript. The cell prints the resulting run prefix, and everything downstream is scoped to it: the Neo4j Browser query at the end of section 5, and the cleanup command you run when you are finished.

The prefix reads `demo08-` because this lab was Demo 08 before the renumber to six labs. It is frozen at that value: records already written to a shared instance under it stop matching a renamed prefix, and nothing in the lab can then find them.

One memory client is opened on first use and closed by the last cell, so the notebook makes one connection rather than four.

In [ ]:
import os
import uuid

import boto3

from memory_helpers import (
    ACTOR_PREFERENCE_RECALL,
    DEMO_ID_PREFIX,
    HERO_HOTEL_NAME,
    WORKSHOP_OWNER,
    build_memory_client,
    get_actor_preferences_for_hotel,
    link_preference_to_message_and_hotel,
    load_config,
    preference_category,
    tag_demo_records,
    titan_access_problem,
)

RUN_ID = uuid.uuid4().hex[:8]
RUN_PREFIX = f"{DEMO_ID_PREFIX}{RUN_ID}-"
ACTOR_A = f"{RUN_PREFIX}guest-alice"
ACTOR_B = f"{RUN_PREFIX}guest-blake"
SESSION_A1 = f"{RUN_PREFIX}session-a1"
SESSION_A2 = f"{RUN_PREFIX}session-a2"
SESSION_B1 = f"{RUN_PREFIX}session-b1"

# One preference category per actor, and that is load-bearing rather than
# tidy. add_preference deduplicates inside a single category at cosine 0.95,
# the two actors write near-paraphrases about the same hotel, and a shared
# category would therefore hand actor B actor A's node. Both categories still
# start with hotels-<run prefix>, which is the handle cleanup uses.
CATEGORY_A = preference_category(RUN_PREFIX, "alice")
CATEGORY_B = preference_category(RUN_PREFIX, "blake")

try:
    config = load_config()
except RuntimeError as exc:
    config = None
    print(f"Neo4j is not configured: {exc}")

RUNNER_SKIP_WRITES = os.getenv("WORKSHOP_RUNNER") == "1"
MEMORY_READY = (
    not RUNNER_SKIP_WRITES
    and config is not None
    and boto3.Session().get_credentials() is not None
)

# Titan Text Embeddings V2 is the third model this workshop needs, and it is
# not the one Lab 1 used. Every memory write embeds its text, so without this
# probe an unenabled model surfaces as a raw AccessDeniedException from inside
# the library at the first write in section 2.
if MEMORY_READY:
    titan_problem = await titan_access_problem(config)
    if titan_problem:
        MEMORY_READY = False
        print(f"Titan Text Embeddings V2 is not usable. {titan_problem}")

MEMORY_CLIENT = None


async def memory_client():
    """Connect the one MemoryClient this notebook uses, then reuse it.

    Connecting is the expensive part of the memory client, so it happens once
    on first use rather than once per section. The last cell closes it.
    """
    global MEMORY_CLIENT
    if MEMORY_CLIENT is None:
        MEMORY_CLIENT = build_memory_client(config)
        await MEMORY_CLIENT.connect()
    return MEMORY_CLIENT


print(f"Run prefix: {RUN_PREFIX}")
if RUNNER_SKIP_WRITES:
    print(
        "Skipping Lab 6 writes: the notebook runner blocks live writes; "
        "pass --allow-writes to enable them."
    )
elif MEMORY_READY:
    print(f"Neo4j at {config.uri}, Bedrock in {config.region}.")
    print("Remove this run alone when you are finished, from 06-memory/:")
    print(
        "    uv run --with-requirements requirements.txt python "
        f"cleanup_memory.py --run-prefix {RUN_PREFIX}"
    )
else:
    print("Not configured. Every live cell below will skip.")

## 2. Persist the preference statement

The core path uses fixture messages instead of another model call. That makes the memory behavior deterministic and keeps this lab focused on Neo4j. The first query also fails with an actionable message if Lab 1 did not create the expected hero Hotel.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    memory = await memory_client()

    hotels = await memory.query.cypher(
        """
        CYPHER 25
        MATCH (h:Hotel {name: $hotel_name})
        RETURN h.name AS name
        """,
        {"hotel_name": HERO_HOTEL_NAME},
    )
    if len(hotels) != 1:
        raise RuntimeError(
            f"Lab 1 must create exactly one Hotel named "
            f"{HERO_HOTEL_NAME!r}; found {len(hotels)}."
        )

    preference_source = await memory.short_term.add_message(
        SESSION_A1,
        "user",
        f"I loved staying at {HERO_HOTEL_NAME}. A room on a high "
        "floor away from the elevator is a must for me.",
        user_identifier=ACTOR_A,
        extraction_mode="skip",
    )
    await memory.short_term.add_message(
        SESSION_A1,
        "assistant",
        "I will remember that hotel and room preference.",
        user_identifier=ACTOR_A,
        extraction_mode="skip",
    )

    print(f"Stored two fixture messages in {SESSION_A1}.")

## 3. Write one explicit preference and its provenance

The library creates the actor-owned `Preference`. Two small workshop-owned relationships make the graph inspectable: `DERIVED_FROM` points to the exact source message and `ABOUT_HOTEL` points directly to the existing Hotel. No `Entity` label or memory property is added to any Hotel node.

A `Preference` carries no session ID or actor identifier of its own, so its `category` is where this run's namespace goes, one category per actor. That is what lets cleanup find a preference whose run died before the tagging cell at the end of this notebook, once the sweeps above it have detached every edge that could otherwise have reached it. Section 4 explains why the two actors need separate categories.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    memory = await memory_client()

    preference = await memory.long_term.add_preference(
        category=CATEGORY_A,
        preference=(
            f"Loves {HERO_HOTEL_NAME} and wants a room on a high "
            "floor away from the elevator."
        ),
        context=f"Workshop run {RUN_ID}, session {SESSION_A1}",
        user_identifier=ACTOR_A,
    )

    linked = link_preference_to_message_and_hotel(
        config,
        str(preference.id),
        str(preference_source.id),
        HERO_HOTEL_NAME,
    )
    if not linked:
        raise RuntimeError(
            "Could not link the preference: the Preference, its source "
            "Message, or exactly one Hotel named "
            f"{HERO_HOTEL_NAME!r} was missing. Re-run section 2, and "
            "confirm Lab 1 built the graph."
        )
    print(f"Preference in category {CATEGORY_A!r}.")
    print("Linked to its source message and the real Hotel.")

## 4. Recall for the same actor, and isolate a second actor

Actor A returns in `SESSION_A2`, a session that did not exist when the preference was written. The recall query takes no session parameter at all, and that is the point rather than an omission: it is anchored on the `User`, so the memory outlives the conversation that produced it and any later session for that actor reaches it.

Actor B is not an empty actor. Blake stays at the same hero hotel and wants the opposite room: ground floor, close to the elevator. So the two actors differ in nothing that a memory store can see. Same hotel, same run, same query shape, one parameter apart. If isolation were only a convention, this is where the wrong row would show up.

It does not, because recall is a traversal that starts at one `User`. A preference the actor does not own is not reachable by the query at all, so there is nothing to filter out and nothing to forget to filter. That is a structural property of the pattern, not a check somewhere in application code. The cell prints the Cypher it runs so the shape is visible.

**The two actors write under separate categories, and that is deliberate.** `add_preference` deduplicates: it looks for an existing preference in the same `category` at cosine 0.95 or above and, on a hit, returns that node and links the calling user to it. Blake's sentence is a near-paraphrase of Alice's about the same hotel, so a shared category is exactly the input that trips it, and Blake would end up holding Alice's row through the library rather than through any failure of the query. The cell asserts the two preferences are distinct nodes, so a regression fails loudly instead of teaching the opposite of the lesson.

In production, the application must still bind these actor and session IDs to authenticated callers. Multi-tenant mode checks that an identifier was supplied. It never checks that the caller is entitled to it.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    memory = await memory_client()

    # Actor A returns in a brand new session and asks.
    await memory.short_term.add_message(
        SESSION_A2,
        "user",
        "What hotel and room preference do you have for me?",
        user_identifier=ACTOR_A,
        extraction_mode="skip",
    )
    # Actor B states the opposite preference about the same hotel.
    blake_source = await memory.short_term.add_message(
        SESSION_B1,
        "user",
        f"I stay at {HERO_HOTEL_NAME} too. Give me a ground-floor "
        "room right by the elevator, I cannot manage stairs.",
        user_identifier=ACTOR_B,
        extraction_mode="skip",
    )
    blake_preference = await memory.long_term.add_preference(
        category=CATEGORY_B,
        preference=(
            f"Loves {HERO_HOTEL_NAME} and wants a ground-floor room "
            "close to the elevator."
        ),
        context=f"Workshop run {RUN_ID}, session {SESSION_B1}",
        user_identifier=ACTOR_B,
    )

    # The regression guard for the deduplication trap described above. A
    # returned node id equal to actor A's means the library matched the two
    # near-paraphrases and linked actor B to actor A's Preference, which the
    # one-row check below would not catch.
    if blake_preference.id == preference.id:
        raise RuntimeError(
            "Actor B was linked to actor A's Preference node. "
            "add_preference deduplicates inside one category at cosine "
            "0.95, so the two actors have to write under separate "
            f"categories: {CATEGORY_A!r} and {CATEGORY_B!r}."
        )

    if not link_preference_to_message_and_hotel(
        config,
        str(blake_preference.id),
        str(blake_source.id),
        HERO_HOTEL_NAME,
    ):
        raise RuntimeError(
            "Could not link actor B's preference to its source message "
            f"and the Hotel named {HERO_HOTEL_NAME!r}."
        )

    print("The recall query, run once per actor:")
    print(ACTOR_PREFERENCE_RECALL)
    print()

    for label, actor in (("Actor A (Alice)", ACTOR_A), ("Actor B (Blake)", ACTOR_B)):
        rows = get_actor_preferences_for_hotel(config, actor, HERO_HOTEL_NAME)
        if len(rows) != 1:
            raise RuntimeError(
                f"{label} should see exactly one preference for "
                f"{HERO_HOTEL_NAME!r}, got {len(rows)}. Isolation or the "
                "section 3 write is wrong."
            )
        print(f"{label}: {len(rows)} row(s)")
        print(f"    {rows[0]['preference']}")

## 5. Inspect the complete provenance path

Section 4 is what any memory store can do. Scope a write to an actor, read it back for that actor, keep the other actor out. A managed service does that, and does it without you owning an embedding contract or a database.

This section is what only a graph can do. The preference is a node sitting in the same graph as the hotels, so it can be walked in both directions from the actor who owns it:

- **Backward**, along `DERIVED_FROM`, to the exact `Message` that produced the preference, and from there along `HAS_MESSAGE` to the `Conversation` it was said in. Not a similar message. The one.
- **Forward**, along `ABOUT_HOTEL`, to the canonical `Hotel` that Lab 1 extracted from the source documents. The same node Lab 2 retrieves against and Lab 4 attaches a reservation request to, not a copy of it and not a string that happens to match.

Standing on that `Hotel`, every other memory about it is one hop back down `ABOUT_HOTEL`. That is the third question the top of this notebook promised, and it is the one a store keyed by actor cannot answer at all: it asks the graph to fan out from a domain node to memories owned by people other than the one who asked.

All of it is one query per question, and the answers are traversals rather than similarity scores. That is the difference between memory you can query and memory you can only retrieve from.

In [ ]:
PROVENANCE_PATH = """
CYPHER 25
MATCH (u:User {identifier: $actor})
      -[:HAS_PREFERENCE]->(p:Preference)
      -[:DERIVED_FROM]->(m:Message)
      <-[:HAS_MESSAGE]-(c:Conversation),
      (p)-[:ABOUT_HOTEL]->(h:Hotel {name: $hotel_name})
RETURN u.identifier AS actor,
       p.preference AS preference,
       m.content AS source_message,
       c.session_id AS source_session,
       h.name AS hotel
"""

# The fan-out the other direction: stand on the Hotel, walk back down
# ABOUT_HOTEL, and every memory about it turns up regardless of who owns it.
# Scoped to this run so a shared instance does not return other participants.
MEMORIES_ABOUT_HOTEL = """
CYPHER 25
MATCH (h:Hotel {name: $hotel_name})
      <-[:ABOUT_HOTEL]-(p:Preference)
      <-[:HAS_PREFERENCE]-(u:User)
WHERE u.identifier STARTS WITH $run_prefix
RETURN u.identifier AS actor, p.preference AS preference
ORDER BY actor
"""


def short(text: str, width: int = 68) -> str:
    """Trim a long property value so the path stays readable."""
    text = " ".join(str(text).split())
    return text if len(text) <= width else f"{text[: width - 1]}..."


if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    memory = await memory_client()

    path_rows = await memory.query.cypher(
        PROVENANCE_PATH,
        {"actor": ACTOR_A, "hotel_name": HERO_HOTEL_NAME},
    )
    fan_out_rows = await memory.query.cypher(
        MEMORIES_ABOUT_HOTEL,
        {"hotel_name": HERO_HOTEL_NAME, "run_prefix": RUN_PREFIX},
    )

    if len(path_rows) != 1:
        raise RuntimeError(
            f"Expected exactly one provenance path for actor A, got "
            f"{len(path_rows)}. Re-run sections 2 and 3."
        )
    row = path_rows[0]

    print("Backward from the actor, to the message that said it:\n")
    print(f"  (:User {{identifier: {short(row['actor'])!r}}})")
    print("       -[:HAS_PREFERENCE]->")
    print(f"  (:Preference {{preference: {short(row['preference'])!r}}})")
    print("       -[:DERIVED_FROM]->")
    print(f"  (:Message {{content: {short(row['source_message'])!r}}})")
    print("       <-[:HAS_MESSAGE]-")
    print(f"  (:Conversation {{session_id: {short(row['source_session'])!r}}})")

    print("\nForward from the same preference, to the Lab 1 Hotel:\n")
    print(f"  (:Preference {{preference: {short(row['preference'])!r}}})")
    print("       -[:ABOUT_HOTEL]->")
    print(f"  (:Hotel {{name: {short(row['hotel'])!r}}})")

    print(
        f"\nAnd back down ABOUT_HOTEL from that Hotel: "
        f"{len(fan_out_rows)} memory record(s) in this run.\n"
    )
    for fan_out in fan_out_rows:
        print(f"  {fan_out['actor']}")
        print(f"       {short(fan_out['preference'])}")

    if len(fan_out_rows) != 2:
        raise RuntimeError(
            f"Expected both actors' preferences to point at "
            f"{HERO_HOTEL_NAME!r}, got {len(fan_out_rows)}."
        )

### See it as a graph

Printed rows are still rows. The point of putting memory in Neo4j is that the subgraph is a picture, so open Neo4j Browser against the same instance and paste in the query the next cell prints. It comes back scoped to your own run prefix, which matters on a shared instance: the bare `demo08-` namespace holds every participant's run and every run anyone has left behind.

Both your actors, both preferences, both source messages, and the one `Hotel` they share, rendered in a single view. The hotel is the node with two preferences hanging off it, and it is the same node Lab 1 extracted, Lab 2 retrieves, and Lab 4 books against. Nothing here is a copy.

Add `<-[:HAS_PREFERENCE]-(:User)` to the pattern to pull the actors in as well.

In [ ]:
# No credentials needed: this prints the query rather than running it.
BROWSER_QUERY = f"""CYPHER 25
MATCH path = (c:Conversation)-[:HAS_MESSAGE]->(:Message)
             <-[:DERIVED_FROM]-(:Preference)-[:ABOUT_HOTEL]->(:Hotel)
WHERE c.session_id STARTS WITH '{RUN_PREFIX}'
RETURN path"""

print("Paste this into Neo4j Browser:\n")
print(BROWSER_QUERY)

## Mark this run for scoped cleanup, and close the client

The IDs already carry the run prefix section 1 printed. The ownership marker provides a second cleanup handle, for records whose run reached this cell. Cleanup deletes only namespaced memory records, orphaned Lab 6 preferences, and workshop-owned relationships; it never changes a Hotel node.

The second cell closes the one memory client the notebook opened.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    marked = tag_demo_records(
        config,
        session_ids=[SESSION_A1, SESSION_A2, SESSION_B1],
        user_identifiers=[ACTOR_A, ACTOR_B],
    )
    print(f"Marked {marked} record(s) with {WORKSHOP_OWNER!r}.")

In [ ]:
if MEMORY_CLIENT is None:
    print("No memory client was opened.")
else:
    await MEMORY_CLIENT.close()
    MEMORY_CLIENT = None
    print("Memory client closed.")

## Choosing a memory architecture

| Dimension | AgentCore Memory | Neo4j graph memory, this lab |
|-----------|------------------|------------------------------|
| How memory is written | Raw events, plus optional managed extraction into long-term records | Explicit application writes, with extraction turned off through `ExtractorType.NONE` |
| When it is recallable | Raw event records immediately; extracted long-term records after asynchronous extraction | Immediately after the write |
| Inspectability | Retrieved through a service API | Queryable graph with source provenance |
| Domain linking | Separate from domain data | Workshop-owned edge to the real `Hotel` |
| Isolation | Actor namespaces managed by the service | Scoped writes and actor-anchored reads; application authorizes sessions |
| Operations | AWS operates the store | You operate Neo4j and the embedding contract |

The row that actually separated them is `Inspectability`, and section 5 is what it means in practice: one traversal from the actor to the exact message that produced the preference, one to the canonical `Hotel`, and one back down to every other memory about that hotel. A store returns the records it holds for an actor. A graph answers questions about how those records connect to the rest of your domain.

Choose AgentCore Memory for managed extraction and managed operations. Choose graph memory when explicit writes, immediate visibility, provenance, and domain relationships matter.

## Remove your run

Cleanup takes the run prefix section 1 printed, so it removes your records and nobody else's. Run it from `06-memory/`, substituting the prefix from your own output:

```bash
uv run --with-requirements requirements.txt python cleanup_memory.py \
    --run-prefix demo08-xxxxxxxx-
```

Add `--dry-run` to see the counts first. Hotel nodes are never touched, and the script counts them before and after to prove it.

Sweeping every Lab 6 run on the instance is a separate `--all` flag behind a typed confirmation. On a shared instance that deletes runs other participants still have in progress, so it is a facilitator action for when the room is finished.

## Where this leaves you

That is the end of the workshop, and memory is the last place its through-line shows up. Neo4j owns the connected data. AWS owns reasoning and hosting. In this lab AWS's share of that is narrow on purpose: Bedrock supplies the memory embeddings and nothing else. Extraction is off, no LLM is constructed, and every memory record here was written explicitly by the notebook rather than inferred by a model.

Every lab put the same boundary in a different place. Lab 1 extracted the graph with Bedrock and stored it in Neo4j. Lab 2 asked the graph questions a vector index alone could not answer. Lab 3 gave a Strands agent a tool instead of a longer prompt. Lab 4 read the guest limit from the graph rather than from an instruction the model could talk its way around. Lab 5 moved that agent onto AgentCore Runtime without changing the retriever. This lab did it once more: the embeddings came from Bedrock, and what the agent remembers went into the same graph as what it knows, joined to it by a real relationship. The preference points at the `Hotel` node Lab 1 built. It did not point at a copy of it, and it did not point at a string.

That is the whole argument. Grounding is not something you add to an agent at the end. It is a decision about where the facts live, made once, and it holds whether the fact is a hotel policy, a reservation rule, or something a guest told you six months ago.

**Taking this further, after the room empties.** The repository stays useful once the workshop instance is gone:

- [Neo4j Aura free tier](https://neo4j.com/cloud/aura-free/) gives you a permanent instance to point the same `.env` at, so every lab here keeps running.
- [`neo4j-agent-memory` on PyPI](https://pypi.org/project/neo4j-agent-memory/) is the library this lab pins at 0.5.0. Its release notes are where to check whether semantic search has gained an owner filter, which is the one thing that would let you use it as an isolation boundary.
- [Strands Agents documentation](https://strandsagents.com) covers tools, hooks, and the [Amazon Bedrock model provider](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/) the agents in Labs 3, 4, and 5 use.
- [`../workshop-delivery/architecture.md`](../workshop-delivery/architecture.md) has the production view: how the pieces fit, which boundaries survive contact with a real deployment, and what changes when the graph is not a workshop instance.

- **Previous:** [Lab 5: Deploy to AgentCore](../05-agentcore-deploy/)
- **Start from the beginning:** [Lab 1: Graph build](../01-graph-build/)